In [ ]:
# @title Zelle 1: Setup, GitHub-Auth, agy CLI & OKF-Skill
import os
import subprocess
import sys

# --- Laufzeit-Check: Cloud-Abhängigkeiten absichern (Colab vs. lokal) ---
try:
    from dotenv import load_dotenv
    load_dotenv()  # lokale .env laden (falls vorhanden)
except ImportError:
    pass  # python-dotenv fehlt (z.B. in Colab); Secrets kommen dann aus google.colab.userdata
IN_COLAB = "google.colab" in sys.modules
WORKDIR = os.environ.get("NEWSCHECK_DIR") or ("/content/newscheck" if IN_COLAB else os.getcwd())

# --- 1. Git & GitHub Auth ---
subprocess.run(["git", "config", "--global", "user.name", "Colab Automation Bot"], check=True)
subprocess.run(["git", "config", "--global", "user.email", "colab-automation-bot@users.noreply.github.com"], check=True)

# --- Dual-Environment Secret Loading ---
if IN_COLAB:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
else:
    GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
    if not GITHUB_TOKEN:
        raise ValueError("CRITICAL: GITHUB_TOKEN is missing from the local .env file!")
os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN

REPO = "rhigazi/newscheck"
CLONE_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"

if os.path.isdir(os.path.join(WORKDIR, ".git")):
    print("Repository vorhanden -> synchronisiere mit origin/main ...")
    # Remote-URL mit aktuellem Token setzen (robust gegen abgelaufene Tokens aus alten Clones)
    subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin", CLONE_URL], check=False)
    # Deterministischer Sync statt 'git pull': robust gegen divergente Branches und
    # lokale Änderungen (vermeidet Merge-Konflikte / 'divergent branches'-Fehler).
    for _cmd in (["fetch", "origin", "main"], ["reset", "--hard", "origin/main"]):
        _r = subprocess.run(["git", "-C", WORKDIR] + _cmd, capture_output=True, text=True, check=False)
        if _r.stdout.strip():
            print(_r.stdout.strip())
        if _r.returncode != 0:
            raise RuntimeError("git " + " ".join(_cmd) + " fehlgeschlagen: " + _r.stderr.strip())
else:
    print("Klone Repository ...")
    _r = subprocess.run(["git", "clone", "--depth", "1", CLONE_URL, WORKDIR], capture_output=True, text=True, check=False)
    if _r.returncode != 0:
        raise RuntimeError("git clone fehlgeschlagen: " + _r.stderr.strip())

os.chdir(WORKDIR)

# --- 2. agy CLI Installation & Pfad-Erweiterung ---
print("Prüfe agy CLI Installation...")
os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.local/bin")
agy_path = subprocess.run(["which", "agy"], capture_output=True, text=True, check=False).stdout.strip()

if not agy_path:
    print("Installiere agy CLI...")
    subprocess.run("curl -fsSL https://antigravity.google/cli/install.sh | bash", shell=True, check=True)

print("Hinweis: Die agy-Authentifizierung wird vom Nutzer im Anschluss manuell durchgeführt.")

# --- 3. OKF Skill non-interaktiv installieren ---
print("Installiere OKF Skill (okf-knowledge-base)...")
env = os.environ.copy()
env["CI"] = "true"  # Zwingt npx/skills in den non-interaktiven Modus

skill_cmd = [
    "npx", "--yes", "skills", "add", 
    "https://github.com/inkeep/open-knowledge-skills", 
    "--skill", "okf-knowledge-base", 
    "--yes"
]
skill_result = subprocess.run(skill_cmd, env=env, capture_output=True, text=True, check=False)
print(skill_result.stdout)
if skill_result.returncode != 0:
    print("Hinweis bei Skill-Installation:", skill_result.stderr)

print("Setup von Repository, agy und OKF-Skill erfolgreich abgeschlossen!")


In [ ]:
# @title Zelle 2: Marktdaten-Abruf (Alpha Vantage / DIA ETF)
import os
import sys

import pandas as pd
import requests
from IPython.display import display

# --- Laufzeit-Check: Cloud-Abhängigkeiten absichern (Colab vs. lokal) ---
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # python-dotenv fehlt (z.B. in Colab); Secrets kommen dann aus google.colab.userdata
IN_COLAB = "google.colab" in sys.modules

# --- Dual-Environment Secret Loading ---
if IN_COLAB:
    from google.colab import userdata
    ALPHA_VANTAGE_API_KEY = userdata.get("ALPHA_VANTAGE_API_KEY")
else:
    ALPHA_VANTAGE_API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")
    if not ALPHA_VANTAGE_API_KEY:
        raise ValueError("CRITICAL: ALPHA_VANTAGE_API_KEY is missing from the local .env file!")
os.environ["ALPHA_VANTAGE_API_KEY"] = ALPHA_VANTAGE_API_KEY

SYMBOL = "DIA"  # SPDR Dow Jones Industrial Average ETF Trust (stabiler Ersatz für DJI-Index)
params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": SYMBOL,
    "outputsize": "compact",
    "apikey": ALPHA_VANTAGE_API_KEY,
}

try:
    resp = requests.get("https://www.alphavantage.co/query", params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
except requests.RequestException as e:
    raise RuntimeError("Alpha Vantage Request fehlgeschlagen: " + str(e))

# --- Rate-Limit / API-Fehler sauber abfangen ---
if "Note" in data or "Information" in data:
    msg = data.get("Note") or data.get("Information")
    raise RuntimeError("Alpha Vantage Rate-Limit erreicht: " + str(msg))
if "Error Message" in data:
    raise RuntimeError("Alpha Vantage Fehler: " + str(data["Error Message"]))
if "Time Series (Daily)" not in data:
    raise RuntimeError("Unerwartete Antwort von Alpha Vantage: " + str(list(data.keys())))

ts = data["Time Series (Daily)"]
df = pd.DataFrame.from_dict(ts, orient="index")
df.index = pd.to_datetime(df.index)
df = df.sort_index()
df.columns = ["open", "high", "low", "close", "volume"]
df = df.astype({"open": float, "high": float, "low": float, "close": float, "volume": int})
df.index.name = "date"

print("DIA-ETF-Tagesdaten erfolgreich geladen:", len(df), "Zeilen")
display(df.tail(10))


In [ ]:
# @title Zelle 3: News-Dateien einlesen (Streng gefiltert: Montag bis Samstag)
import os
import sys

# --- Laufzeit-Check: Colab-Pfad vs. lokaler Pfad (konsistent zu Zelle 4/5) ---
IN_COLAB = "google.colab" in sys.modules
WORKDIR = os.environ.get("NEWSCHECK_DIR") or ("/content/newscheck" if IN_COLAB else os.getcwd())

# Erlaubte Wochentage definieren
ERLAUBTE_TAGE = ["montag.txt", "dienstag.txt", "mittwoch.txt", "donnerstag.txt", "freitag.txt", "samstag.txt"]

# Gezieltes Einlesen nur der erlaubten Wochentags-Dateien
news_archive = {}
for tag in ERLAUBTE_TAGE:
    path = os.path.join(WORKDIR, tag)
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            news_archive[tag] = f.read()

if not news_archive:
    raise FileNotFoundError("Keine g\u00fcltigen Wochentags-Dateien (montag.txt bis samstag.txt) in " + WORKDIR + " gefunden.")

print(f"\u2705 {len(news_archive)} Wochentags-Dateien erfolgreich geladen:")
for _name, _text in news_archive.items():
    print(f"  - {_name}: {len(_text)} Zeichen")


In [ ]:
# @title Zelle 3.5: agy-Authentifizierung (manuell)
import os
import subprocess

os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.local/bin")

print("Teste agy-Authentifizierung mit 'agy -p \"Test\"' ...")
result = subprocess.run(
    ["agy", "-p", "Test"],
    capture_output=True,
    text=True,
    check=False,
    stdin=subprocess.DEVNULL,
)

print("stdout:", result.stdout)
if result.returncode != 0:
    print("stderr:", result.stderr)
    print("\u27a1\ufe0f  agy ist noch nicht authentifiziert. F\u00fchre den manuellen Login durch und starte diese Zelle erneut.")
else:
    print("\u2705 agy authentifiziert und bereit f\u00fcr die Analyse.")


In [ ]:
# @title Zelle 4: Prognose-Generierung & OKF-Regelsynthese via agy (OKF-Skill)
import os
import subprocess
import sys
import textwrap

# --- Laufzeit-Check: Colab-Pfad vs. lokaler Pfad ---
IN_COLAB = "google.colab" in sys.modules
WORKDIR = os.environ.get("NEWSCHECK_DIR") or ("/content/newscheck" if IN_COLAB else os.getcwd())

# --- Token-Optimierung: News-Texte vor der LLM-Übergabe begrenzen ---
MAX_CHARS_PER_FILE = 4000
MAX_TOTAL_CHARS = 20000

def _truncate(text, limit):
    text = text.strip()
    return text[:limit] + ("\n...[gekürzt]" if len(text) > limit else "")

_snippets = []
_total = 0
for _name, _text in news_archive.items():
    if _total >= MAX_TOTAL_CHARS:
        break
    _snippet = _truncate(_text, MAX_CHARS_PER_FILE)
    _snippets.append("### " + _name + "\n" + _snippet)
    _total += len(_snippet)

news_block = "\n\n".join(_snippets)
market_block = df.tail(10).to_string()

# --- Termine (Wochenvorschau) einlesen ---
_termine_path = os.path.join(WORKDIR, "termine.md")
if os.path.exists(_termine_path):
    with open(_termine_path, encoding="utf-8", errors="replace") as _f:
        termine_block = _f.read()[:4000]
else:
    termine_block = "(keine termine.md vorhanden)"

# --- Vorige Erwartung für die Soll-Ist-Validierung (Zelle 4.5) sichern ---
_erw_path = os.path.join(WORKDIR, "erwartung.md")
ERWARTUNG_VORIGE_TEXT = None
if os.path.exists(_erw_path):
    with open(_erw_path, encoding="utf-8", errors="replace") as _f:
        ERWARTUNG_VORIGE_TEXT = _f.read()

prompt = textwrap.dedent(f"""
    Nutze den installierten OKF-Skill (okf-knowledge-base) gemäß Open Knowledge Format (OKF) v0.2.

    Aufgabe 1 - OKF-Regelsynthese: Analysiere die menschlichen Verhaltensnarrative
    (z. B. Panik, Zinsangst, Erwartungshaltungen) in den Nachrichten und gleiche sie
    mit den DJIA-Kursdaten (DIA-ETF als Proxy) ab. Synthetisiere daraus OKF-Regeln und
    lege/aktualisiere die OKF-Dateien im Verzeichnis ./okf-regeln/ gemäß OKF v0.2.

    Aufgabe 2 - Prognose (erwartung.md): Erzeuge ODER aktualisiere zwingend die Datei
    "erwartung.md" im Stammverzeichnis mit einer strukturierten, tagesaktuellen
    Soll-Prognose für die anstehenden Termine. Diese Datei MUSS mit folgendem
    YAML-Frontmatter beginnen (exakte Schlüssel, maschinenlesbar):

    ---
    okf_version: "0.2"
    typ: erwartung
    datum: <YYYY-MM-DD heute>
    bezugsregel: <Dateiname der tragenden OKF-Regel ohne .md, z.B. regel-pre-event-paralyse>
    richtung: <bullisch | baerisch | neutral>
    spanne_untere: <erwarteter Unterkurs als Zahl, z.B. 508.00>
    spanne_obere: <erwarteter Oberkurs als Zahl, z.B. 516.00>
    volumen: <hoch | normal | niedrig>
    konfidenz: <Zahl 0.0-1.0>
    ---

    Danach folgt eine Prosa-Begründung inkl. der ausgewerteten Termine.

    ## Anstehende Termine (termine.md)
    {termine_block}

    ## Marktdaten (DJIA/DIA, letzte 10 Handelstage)
    {market_block}

    ## Nachrichten-Snippets (token-optimiert)
    {news_block}
""").strip()

# --- agy (Google Antigravity CLI) im Headless-/Print-Modus ausführen ---
# WICHTIG: agy ist flag-basiert (KEIN 'run'-Subcommand). --print muss das LETZTE Flag sein.
cmd = [
    "agy",
    "--dangerously-skip-permissions",
    "--add-dir", WORKDIR,
    "--print-timeout", "20m",
    "--print", prompt,
]
print("Starte agy-Prognose & OKF-Regelsynthese ...")
result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    check=False,
    cwd=WORKDIR,
    stdin=subprocess.DEVNULL,
)
print(result.stdout)
if result.returncode != 0:
    print("agy stderr:", result.stderr)
    raise RuntimeError("agy-Analyse fehlgeschlagen")


In [ ]:
# @title Zelle 4.5: Soll-Ist-Validierung & Feedback (Regelkorrektur)
import datetime
import os
import sys

# --- Laufzeit-Check: Colab-Pfad vs. lokaler Pfad ---
IN_COLAB = "google.colab" in sys.modules
WORKDIR = os.environ.get("NEWSCHECK_DIR") or ("/content/newscheck" if IN_COLAB else os.getcwd())
OKF_DIR = os.path.join(WORKDIR, "okf-regeln")
LOG_PATH = os.path.join(OKF_DIR, "log.md")


def _frontmatter(text):
    """Extrahiert einfache 'schluessel: wert'-Paare aus dem YAML-Frontmatter (ohne pyyaml)."""
    d = {}
    lines = text.splitlines()
    if not lines or lines[0].strip() != "---":
        return d
    for line in lines[1:]:
        if line.strip() == "---":
            break
        if ":" in line:
            k, v = line.split(":", 1)
            d[k.strip().lower()] = v.strip().strip('"').strip("'")
    return d


def _adjust_konfidenz(path, delta):
    """Passt das konfidenz-Feld einer OKF-Regel an (Delta, gekappt auf 0.05..0.99)."""
    if not path or not os.path.exists(path):
        return False
    with open(path, encoding="utf-8") as _f:
        lines = _f.read().splitlines(keepends=True)
    idx = None
    for i, line in enumerate(lines):
        if line.strip().lower().startswith("konfidenz:"):
            idx = i
            break
    if idx is not None:
        try:
            old = float(lines[idx].split(":", 1)[1].strip())
        except ValueError:
            old = 0.5
        new = max(0.05, min(0.99, round(old + delta, 2)))
        lines[idx] = "konfidenz: " + format(new, ".2f") + "\n"
    else:
        insert_at = 1 if lines and lines[0].strip() == "---" else 0
        new = max(0.05, min(0.99, 0.5 + delta))
        lines.insert(insert_at, "konfidenz: " + format(new, ".2f") + "\n")
    with open(path, "w", encoding="utf-8") as _f:
        _f.write("".join(lines))
    return True


def _annotate_rule(path, note):
    if not path or not os.path.exists(path):
        return
    with open(path, "a", encoding="utf-8") as f:
        f.write("\n> Korrektur: " + note + "\n")


def _append_log(entry):
    os.makedirs(OKF_DIR, exist_ok=True)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(entry)


# --- Gestern-Erwartung laden (von Zelle 4 gesichert, sonst von Disk) ---
prev_text = globals().get("ERWARTUNG_VORIGE_TEXT")
if prev_text is None:
    _p = os.path.join(WORKDIR, "erwartung.md")
    if os.path.exists(_p):
        with open(_p, encoding="utf-8", errors="replace") as _f:
            prev_text = _f.read()

if "df" not in globals():
    print("Marktdaten (df) fehlen – bitte zuerst Zelle 2 ausführen. Validierung übersprungen.")
elif not prev_text:
    print("Keine vorige erwartung.md gefunden – erster Lauf, Validierung übersprungen.")
else:
    exp = _frontmatter(prev_text)

    # --- Ist-Werte aus den frischen DIA-Marktdaten (Zelle 2) ---
    dfv = df.sort_index()
    heute = dfv.iloc[-1]
    gestern = dfv.iloc[-2]
    close_h = float(heute["close"])
    close_g = float(gestern["close"])
    vol_h = float(heute["volume"])
    mean_vol = float(dfv["volume"].tail(20).mean())

    if close_h > close_g * 1.0005:
        ist_richtung = "bullisch"
    elif close_h < close_g * 0.9995:
        ist_richtung = "baerisch"
    else:
        ist_richtung = "neutral"

    ratio = vol_h / mean_vol if mean_vol else 1.0
    ist_volumen = "hoch" if ratio > 1.3 else ("niedrig" if ratio < 0.7 else "normal")

    # --- Soll-Werte aus der Erwartung ---
    exp_richtung = exp.get("richtung", "").lower()
    exp_volumen = exp.get("volumen", "").lower()
    try:
        spanne_u = float(exp.get("spanne_untere", "nan"))
    except ValueError:
        spanne_u = float("nan")
    try:
        spanne_o = float(exp.get("spanne_obere", "nan"))
    except ValueError:
        spanne_o = float("nan")

    # --- Soll-Ist-Abgleich ---
    t_richtung = (exp_richtung == ist_richtung)
    t_spanne = (spanne_u <= close_h <= spanne_o)
    t_volumen = (exp_volumen == ist_volumen)
    hits = int(t_richtung) + int(t_spanne) + int(t_volumen)
    is_hit = (hits >= 2)

    print("=== Soll-Ist-Validierung ===")
    print("  Richtung  Soll=" + exp_richtung + " | Ist=" + ist_richtung + " -> " + ("TREFFER" if t_richtung else "ABWEICHUNG"))
    print("  Spanne    Soll=[" + str(spanne_u) + ", " + str(spanne_o) + "] | Ist(close)=" + str(close_h) + " -> " + ("TREFFER" if t_spanne else "ABWEICHUNG"))
    print("  Volumen   Soll=" + exp_volumen + " | Ist=" + ist_volumen + " -> " + ("TREFFER" if t_volumen else "ABWEICHUNG"))

    bezug = exp.get("bezugsregel", "")
    rule_path = os.path.join(OKF_DIR, bezug + ".md") if bezug else None

    if is_hit:
        _adjust_konfidenz(rule_path, +0.05)
        print("Ergebnis: TREFFER (" + str(hits) + "/3) -> Konfidenz der Regel '" + bezug + "' gestärkt.")
    else:
        ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
        entry = (
            "\n## Audit " + ts + "\n"
            "- Erwartung (" + exp.get("datum", "?") + "): Richtung=" + exp_richtung
            + ", Spanne=[" + str(spanne_u) + ", " + str(spanne_o) + "], Volumen=" + exp_volumen
            + ", Konfidenz=" + exp.get("konfidenz", "?") + "\n"
            "- Ist: Richtung=" + ist_richtung + ", Close=" + str(close_h)
            + ", Volumen=" + ist_volumen + " (Ratio " + format(ratio, ".2f") + ")\n"
            "- Treffer (" + str(hits) + "/3): Richtung=" + str(t_richtung)
            + ", Spanne=" + str(t_spanne) + ", Volumen=" + str(t_volumen) + "\n"
            "- Massnahme: Konfidenz der Regel '" + bezug + "' gesenkt, Regel korrigiert.\n"
        )
        _append_log(entry)
        _adjust_konfidenz(rule_path, -0.05)
        _annotate_rule(rule_path, ts + " Soll-Ist-Abweichung (" + str(hits) + "/3 Treffer) – Regel hinterfragt.")
        print("Ergebnis: signifikante ABWEICHUNG (" + str(hits) + "/3) -> Audit in log.md + Regelkorrektur.")


In [ ]:
# @title Zelle 5: Automatisierter Git Commit & Push
import datetime
import os
import subprocess
import sys

# --- Laufzeit-Check: Colab-Pfad vs. lokaler Pfad ---
IN_COLAB = "google.colab" in sys.modules
WORKDIR = os.environ.get("NEWSCHECK_DIR") or ("/content/newscheck" if IN_COLAB else os.getcwd())
os.chdir(WORKDIR)


def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True, check=False)
    if r.returncode != 0:
        raise RuntimeError("git " + " ".join(args) + " fehlgeschlagen: " + (r.stderr.strip() or r.stdout.strip()))
    return r


# --- Idempotenz: nur bei echten Änderungen committen ---
# git add -A erfasst erwartung.md, okf-regeln/log.md sowie korrigierte OKF-Regeln automatisch.
status = _git("status", "--porcelain").stdout.strip()
if not status:
    print("Keine Änderungen erkannt -> nichts zu committen (idempotent).")
else:
    print("Änderungen erkannt:")
    print(status)
    _git("add", "-A")
    _stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    _git("commit", "-m", "chore(okf): Regelupdate & Feedback-Loop (erwartung.md/log.md) (" + _stamp + ")")

    # --- Vor dem Push mit dem neuesten origin/main synchronisieren ---
    # Robust gegen parallele Änderungen (z.B. Code-Updates auf origin): den lokalen
    # Commit auf den neuesten Stand von origin/main rebasen (Rebase-Editor deaktiviert),
    # dann fast-forward pushen. Zeigt im Fehlerfall die echte Git-Meldung an.
    _git("fetch", "origin", "main")
    _env = os.environ.copy()
    _env["GIT_EDITOR"] = "true"
    _env["GIT_SEQUENCE_EDITOR"] = "true"
    rebase = subprocess.run(["git", "rebase", "origin/main"], capture_output=True, text=True, check=False, env=_env)
    if rebase.returncode != 0:
        subprocess.run(["git", "rebase", "--abort"], capture_output=True, check=False)
        raise RuntimeError("Push nicht möglich (Rebase-Konflikt mit origin/main): " + rebase.stderr.strip())

    push = _git("push", "origin", "main")
    print("Push erfolgreich:", push.stdout or "ok")


In [ ]:
# @title Zelle 6: Telegram-Benachrichtigung (Abschluss)
import datetime
import os
import sys

import requests

# --- Laufzeit-Check & Secrets (Dual-Environment wie in Zelle 1/2) ---
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # python-dotenv fehlt (z.B. in Colab); Secrets kommen dann aus google.colab.userdata
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
    TELEGRAM_CHAT_ID = userdata.get("TELEGRAM_CHAT_ID")
else:
    TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
    TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID")

if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
    print("Telegram-Secrets fehlen (TELEGRAM_BOT_TOKEN / TELEGRAM_CHAT_ID) – Benachrichtigung übersprungen.")
else:
    # --- Kurze Zusammenfassung für die Nachricht ---
    WORKDIR = os.environ.get("NEWSCHECK_DIR") or ("/content/newscheck" if IN_COLAB else os.getcwd())
    _okf_dir = os.path.join(WORKDIR, "okf-regeln")
    _okf_count = len([f for f in os.listdir(_okf_dir) if f.endswith(".md")]) if os.path.isdir(_okf_dir) else 0
    _erwartung = "erstellt/aktualisiert" if os.path.exists(os.path.join(WORKDIR, "erwartung.md")) else "fehlt"
    _rows = len(df) if "df" in globals() else "?"
    _stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

    text = f"""✅ Newscheck-Pipeline abgeschlossen ({_stamp})

• Marktdaten (DIA): {_rows} Zeilen
• OKF-Regeln: {_okf_count} Dateien in okf-regeln/
• erwartung.md: {_erwartung}
• Push nach origin/main: siehe vorherige Zelle"""

    # --- Telegram senden (best effort; Versandfehler sind nicht fatal) ---
    _url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    try:
        _resp = requests.post(_url, data={"chat_id": TELEGRAM_CHAT_ID, "text": text}, timeout=30)
        if _resp.status_code == 200 and _resp.json().get("ok"):
            print("Telegram-Nachricht gesendet ✓")
        else:
            print("Telegram-Versand fehlgeschlagen:", _resp.status_code, _resp.text)
    except requests.RequestException as e:
        print("Telegram-Versand fehlgeschlagen (Netzwerkfehler):", e)
